In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, average_precision_score,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from DataProcessing import (
    Sensors, Actuators,
    make_sequences, make_sequence_end_timestamps, mixed_scale_features,
)
from ModelTraining import LSTMAutoencoder
from MahalanobisDetector import WhitenedPCAMahalanobisDetector

device = torch.device(
    'mps'  if torch.backends.mps.is_available()  else
    'cuda' if torch.cuda.is_available()           else
    'cpu'
)
print('Device:', device)

In [ ]:
SEQ_LEN    = 64
STRIDE     = 5
INPUT_DIM  = 51
BATCH_SIZE = 512
START_TIME = pd.to_datetime('2015-12-23 12:00:00')

MODEL_PARAMS = dict(
    conv_channels=64,
    kernel_size=3,
    hidden_size=64,
    num_layers=4,
    dropout=0.49521204489910164,
    bidirectional=True,
)

RESULT_DIR     = ROOT / 'results'
DATASET_DIR    = ROOT.parent / 'Dataset'
MODEL_PATH     = RESULT_DIR / 'conv_bilstm_autoencoder.pt'
REF_EVAL_PATH  = RESULT_DIR / 'evaluation_df_with_rules.parquet'
OUT_EVAL_PATH  = RESULT_DIR / 'evaluation_df_with_wpca.parquet'
OUT_TABLE_PATH = RESULT_DIR / 'table_wpca_evaluation.csv'

#### Load Reference Evaluation (labels + rule predictions)

In [ ]:
ref_df  = pd.read_parquet(REF_EVAL_PATH).reset_index(drop=True)
y_dyn   = ref_df['label_dynamic'].to_numpy(dtype='int64')
y_off   = ref_df['label'].to_numpy(dtype='int64')
rule_hc = ref_df['rule_high_conf'].to_numpy(dtype='int64')

print(f'Windows: {len(ref_df)}  '
      f'Positives (dynamic): {y_dyn.sum()}  '
      f'Positives (official): {y_off.sum()}')

#### Build Sequences

In [ ]:
def prepare_sequences(seq_len, stride):
    df_normal = pd.read_parquet(DATASET_DIR / 'SWaT_Dataset_Normal_v1.parquet')
    df_attack = pd.read_parquet(DATASET_DIR / 'SWaT_Dataset_Attack_v1.parquet')
    df_normal['Timestamp'] = pd.to_datetime(df_normal['Timestamp'])
    df_attack['Timestamp'] = pd.to_datetime(df_attack['Timestamp'])
    df_normal = df_normal.loc[df_normal['Timestamp'] >= START_TIME].copy()

    train_set, other = train_test_split(df_normal, train_size=0.8, shuffle=False)
    val_set,   tmp   = train_test_split(other,     train_size=0.5, shuffle=False)
    test_set = pd.concat([tmp, df_attack], axis=0, ignore_index=True)

    feat = Sensors + Actuators
    _, val_x, test_x, _, _ = mixed_scale_features(train_set, val_set, test_set, feat)

    X_val,  _, _ = make_sequences(val_x,  val_set['Label'].to_numpy(),
                                  seq_len, stride, val_set['Timestamp'].to_numpy())
    X_test, _, _ = make_sequences(test_x, test_set['Label'].to_numpy(),
                                  seq_len, stride, test_set['Timestamp'].to_numpy())
    ts_end, _    = make_sequence_end_timestamps(
                       test_set['Timestamp'].to_numpy(), seq_len, stride)

    return X_val, X_test, pd.to_datetime(pd.Series(ts_end)).reset_index(drop=True)


X_val, X_test, ts_end = prepare_sequences(SEQ_LEN, STRIDE)

assert len(ts_end) == len(ref_df), \
    f'Window count mismatch: {len(ts_end)} vs {len(ref_df)}'
assert pd.to_datetime(ref_df['window_end']).equals(ts_end), \
    'window_end timestamps do not match'

print(f'X_val:  {X_val.shape}')
print(f'X_test: {X_test.shape}')

#### Load Model + Fit WhitenedPCA-Mahalanobis Detector

In [ ]:
model = LSTMAutoencoder(input_dim=INPUT_DIM, **MODEL_PARAMS)
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model = model.to(device)

detector = WhitenedPCAMahalanobisDetector(
    model, device,
    n_components=22,
    support_fraction=0.925,
)
detector.fit(X_val)
print('Detector fitted on validation windows.')

#### Score Test Windows

In [ ]:
wpca_scores = detector.score(X_test)
threshold   = detector.threshold_percentile(99.2)

y_wpca    = (wpca_scores > threshold).astype('int64')
y_wpca_or = ((y_wpca == 1) | (rule_hc == 1)).astype('int64')

print(f'Threshold (val pct=99.2): {threshold:.3f}')
print(f'Positive predictions: {y_wpca.sum()} (AE only)  /  {y_wpca_or.sum()} (+ rule OR)')

#### Evaluation Results

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1-score':  f1_score(y_true, y_pred, zero_division=0),
        'AUC-PR':    average_precision_score(y_true, y_pred.astype(float))
                     if len(np.unique(y_true)) > 1 else float('nan'),
        'Positives': int(y_pred.sum()),
    }


rows = []
for col in ['y_pred', 'hybrid_adaptive_f1']:
    if col in ref_df.columns:
        rows.append({'Setting': f'baseline_{col}',
                     **compute_metrics(y_dyn, ref_df[col].to_numpy(dtype='int64'))})

rows.append({'Setting': 'WhitenPCA-Maha pct99.2 + rule OR  [label_dynamic]',
             **compute_metrics(y_dyn, y_wpca_or)})
rows.append({'Setting': 'WhitenPCA-Maha pct99.2 + rule OR  [official_label]',
             **compute_metrics(y_off, y_wpca_or)})
rows.append({'Setting': 'WhitenPCA-Maha pct99.2 only       [label_dynamic]',
             **compute_metrics(y_dyn, y_wpca)})

result_df = pd.DataFrame(rows)
result_df[['Precision','Recall','F1-score','AUC-PR']] = \
    result_df[['Precision','Recall','F1-score','AUC-PR']].round(4)

display(result_df)

#### Save Results

In [ ]:
ref_df['wpca_score'] = wpca_scores
ref_df['y_wpca']     = y_wpca
ref_df['y_wpca_or']  = y_wpca_or

ref_df.to_parquet(OUT_EVAL_PATH, index=False)
result_df.to_csv(OUT_TABLE_PATH, index=False)

print(f'Saved: {OUT_EVAL_PATH}')
print(f'Saved: {OUT_TABLE_PATH}')